最终的代码，采取分数最高的V13版本

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from tqdm import tqdm
import datetime
import gc

# 设置全局绘图风格
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_context("notebook", font_scale=1.2)

def log(msg):
    print(f"[{datetime.datetime.now().strftime('%H:%M:%S')}] {msg}")

# ====================================================
# 1. 配置与数据加载 (内存优化型)
# ====================================================
log("Step 1: Loading Data...")
BASE = "/kaggle/input/competitions/h-and-m-personalized-fashion-recommendations/"

# 采用特定字段读取，避免加载无用信息占用内存
articles = pd.read_csv(BASE + "articles.csv", dtype={'article_id': str})
trans = pd.read_csv(BASE + "transactions_train.csv",
                    dtype={'article_id': str, 'customer_id': str},
                    parse_dates=['t_dat'])

# H&M 比赛的核心在于“时效性”，我们只取最后 31 天的数据进行建模
max_date = trans['t_dat'].max()
df = trans[trans['t_dat'] > max_date - pd.Timedelta(days=31)].copy()
del trans # 及时释放内存
gc.collect()

# ====================================================
# 2. 深度可视化分析 (模仿同学代码的亮点)
# ====================================================
log("Step 2: Visualizing Data Insights...")

def plot_insights(df, articles):
    fig, axes = plt.subplots(2, 2, figsize=(22, 12))

    # 图1：最近一个月的销售热度波动 (趋势分析)
    daily_sales = df.groupby('t_dat').size()
    axes[0, 0].plot(daily_sales.index, daily_sales.values, color='#ff7f0e', lw=3)
    axes[0, 0].set_title("Daily Transaction Volume (Last 31 Days)", fontsize=16)

    # 图2：热门品类分布 (业务理解)
    top_items_info = df['article_id'].value_counts()[:10].index
    top_items_names = articles[articles['article_id'].isin(top_items_info)]['prod_name'].values
    sns.barplot(x=df['article_id'].value_counts()[:10].values, y=top_items_names, ax=axes[0, 1], palette="magma")
    axes[0, 1].set_title("Top 10 Selling Products", fontsize=16)

    # 图3：用户购买件数分布 (用户粘性)
    user_buy_counts = df.groupby('customer_id').size()
    sns.histplot(user_buy_counts, bins=50, ax=axes[1, 0], color='#2ca02c', log_scale=(False, True))
    axes[1, 0].set_title("Purchase Count Distribution (Log Scale)", fontsize=16)

    # 图4：价格偏好分析
    sns.boxenplot(x=df['price'], ax=axes[1, 1], color='#d62728')
    axes[1, 1].set_title("Price Distribution Analysis", fontsize=16)

    plt.tight_layout()
    plt.show()

plot_insights(df, articles)

# ====================================================
# 3. 核心算法：带权复购规则 (V13 Logic)
# ====================================================
log("Step 3: Building Rule-Based Recommender...")

# A. 计算时间衰减权重：离今天越近的购买，权重越高
# 逻辑：快时尚行业，两周前的喜好可能已经变了，赋予最近行为更高的权重
df['days_since'] = (max_date - df['t_dat']).dt.days
df['weight'] = np.exp(-df['days_since'] * 0.1) # 指数衰减

# B. 预计算全局最热单品 (作为补位)
# 这里的热门也是带权重的，保证推的是“正在流行”的单品
popular_items = df.groupby('article_id')['weight'].sum().sort_values(ascending=False).index[:12].tolist()

# C. 构造用户购买行为记录
# 我们需要按时间顺序排列，越近买的越排在后面
user_history = df.sort_values('t_dat').groupby('customer_id')['article_id'].apply(list).to_dict()

def get_recommendation(u_id):
    """V13 核心：加权复购策略"""
    history = user_history.get(u_id, [])

    if not history:
        return " ".join(popular_items)

    # 建立候选人打分表
    candidate_scores = Counter()
    n = len(history)

    for i, item in enumerate(history):
        # 1. 基础权重：位置权重（越靠后越新，权重越高）
        # 2. 累加效应：买过多次的单品分数自然更高
        candidate_scores[item] += (i + 1) / n

    # 取分数最高的 12 个作为预测
    top_12 = [x[0] for x in candidate_scores.most_common(12)]

    # 补位逻辑：如果买过的单品不足 12 个，用全局热门单品填满
    if len(top_12) < 12:
        for p in popular_items:
            if p not in top_12:
                top_12.append(p)
            if len(top_12) >= 12:
                break

    return " ".join(top_12)

# ====================================================
# 4. 生成提交 (高效批处理)
# ====================================================
log("Step 4: Generating Submission File...")
sub = pd.read_csv(BASE + "sample_submission.csv")

# 使用 progress_apply 监控进度，这对大数据处理非常重要
tqdm.pandas()
sub['prediction'] = sub['customer_id'].progress_apply(get_recommendation)

sub.to_csv("submission_v13_complete.csv", index=False)
log("Success: V13 Complete Run Finished!")